In [0]:
# 1. Input Data Prep - Categorical --> One-hot, Repetition, any processing
# 2. Train, val, test, back testing split
# 3. Feature Selection
# 3.1 Variance-based
# 3.2 Mutual Info (Optional)
# 3.3.1 XGB/LGBM Regressor - top n features
# 3.3.2 TweedieRegressor - top n features
# 3.3.3 BorutaPy - top n features
# 3.3.4 ElasticNetCV

In [0]:
from pyspark.sql.types import (
    StringType, DoubleType, LongType, DateType,
    IntegerType, FloatType, ArrayType, DecimalType
)

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/OE/Kerendia_OE_2.0/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/OE/Kerendia_OE_2.0/00_Helper_Notebooks/00_config"

In [0]:
train_data = spark.read \
    .format("delta") \
    .load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/final_mdb")

In [0]:
pd.DataFrame(train_data.columns).display()

In [0]:
print("shape : ",train_data.count(), len(train_data.columns))

In [0]:
# # Loading Training Data
# train_data = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/dev_folder/notebook_output/kerendia_nba_3_hcp_model_train_data.parquet")

# train_data = train_data.drop('PULL_DT')
# print("Shape :", train_data.count(), len(train_data.columns))

#### Drop Features due to Low Variance/Feature being captured multiple times/Feature has very high number string classes

In [0]:
# removing census features
cols_to_drop = ['BH_ID', 'BHID', 'PROVIDER_ID', 'SVC_DT_M', 'COHORT', 'KAISER', 'CANDIDATE_RECORD', 'FELLOW', 'IS_EXTERNALLY_MASTERED', 'IS_VEEVA_MASTER', 'GRAD_YEAR', 'GRAD_TRG_END_DATE', 'PRIMARY_SPECIALTY_GROUP', 'ACCT_NAME', 'MAJOR_CLASS_OF_TRADE', 'DESCRIPTION', 'ZIP_CODE_GEO', 'TERR_CD_GEO', 'CITY_GEO', 'ZIPCODE']

feature_columns = train_data.columns

census_features = [col for col in feature_columns if col.startswith("ACS_PCT_")]
print(len(census_features))

from collections import OrderedDict

to_drop = list(OrderedDict.fromkeys((census_features + cols_to_drop)))

train_data = train_data.drop(*to_drop)
print("Shape :", train_data.count(), len(train_data.columns))

In [0]:
feature_columns = train_data.columns
census_features = [col for col in feature_columns if col.startswith("ACS_PCT_")]
print(len(census_features))
to_drop = list(OrderedDict.fromkeys(census_features))

In [0]:
categoricalcols = [f.name for f in train_data.schema.fields if isinstance(f.dataType, StringType )]
DoubleTypescols = [f.name for f in train_data.schema.fields if isinstance(f.dataType, DoubleType) ]
LongTypecols = [f.name for f in train_data.schema.fields if isinstance(f.dataType, LongType) ]
DateTypecols = [f.name for f in train_data.schema.fields if isinstance(f.dataType, DateType )]
IntTypecols = [f.name for f in train_data.schema.fields if isinstance(f.dataType, IntegerType )]
FloatTypecols = [f.name for f in train_data.schema.fields if isinstance(f.dataType, FloatType )]
ArrayTypecols = [f.name for f in train_data.schema.fields if isinstance(f.dataType, ArrayType )]
DecimalTypecols = [f.name for f in train_data.schema.fields if isinstance(f.dataType, DecimalType )]
NumericTypecols = [f.name for f in train_data.schema.fields if not isinstance(f.dataType, StringType ) | isinstance(f.dataType, DateType ) | isinstance(f.dataType, ArrayType) ]

# All columns except array and datetype
Allcols = [f.name for f in train_data.schema.fields if not isinstance(f.dataType, DateType ) | isinstance(f.dataType, ArrayType) ]


# floatcols = [item[0] for item in combined_data.dtypes if item[1].startswith('integer') ]
print("Length of Categorical features:",len(categoricalcols))
# print(categoricalcols)

print("Length of Double Type features:", len(DoubleTypescols))
# print(continuouscols)

print("Length of Long Type features:", len(LongTypecols))
# print(LongTypecols)

print("Length of Date Type features:", len(DateTypecols))
# print(DateTypecols)

print("Length of Int Type features:", len(IntTypecols))
# print(IntTypecols)

print("Length of Float Type features:", len(FloatTypecols))
#print(FloatTypecols)

print("Length of Array Type features:", len(ArrayTypecols))
# print(ArrayTypecols)

print("Length of Decimal Type features:", len(DecimalTypecols))
# print(DecimalTypecols)

print("Length of Numerical Type features:", len(NumericTypecols))
# print(NumericTypecols)

print("Length of All features:", len(Allcols))
# print(NumericTypecols)

In [0]:
categoricalcols

In [0]:
key_cols = ['INSTANCE','COHORT','HCP_COHORT_ID','PROVIDER_ID']

train_data_wo_cat = train_data.drop(*categoricalcols)
train_data_wo_cat = train_data_wo_cat.drop(*key_cols)
print("Shape :", train_data_wo_cat.count(), len(train_data_wo_cat.columns))

#### Step 1: Variance-based

In [0]:
# Step 1.1: Remove low variance
stddevs = train_data_wo_cat.select([stddev(col(c)).alias(c) for c in train_data_wo_cat.columns if c != "TOTAL_NBRX_C3" and dict(train_data_wo_cat.dtypes)[c] in ('int', 'double', 'float')])
stddev_dict = stddevs.collect()[0].asDict()

non_zero_std_cols = [c for c, std in stddev_dict.items() if std is not None and std > 1e-6]
print("Number of columns with non-zero standard deviation:", len(non_zero_std_cols))
train_data_wo_cat = train_data_wo_cat.select(non_zero_std_cols + ['TOTAL_NBRX_C3'])
print("Shape :", train_data_wo_cat.count(), len(train_data_wo_cat.columns))

In [0]:
pd.DataFrame(non_zero_std_cols).display()

In [0]:
# Step 1.2: Checking Null Rate - No feature is removed from this step
row_count = train_data_wo_cat.count()
null_ratios = train_data_wo_cat.select([(count(when(col(c).isNull(), c)) / row_count).alias(c) for c in train_data_wo_cat.columns])
nulls = null_ratios.collect()[0].asDict()
pd.DataFrame(list(nulls.items()), columns=['column_name', 'null_ratio']).sort_values(by='null_ratio', ascending=False).display()

#### Step 2: Mutual Information

In [0]:
# train_data_wo_cat_sampled = train_data_wo_cat.sample(withReplacement=False, fraction=0.1, seed=42)
# print("Shape :", train_data_wo_cat_sampled.count(), len(train_data_wo_cat_sampled.columns))

In [0]:
from sklearn.feature_selection import mutual_info_regression

# Only use non-zero TOTAL_NBRX_C3s for more meaningful info
non_zero_train_data = train_data_wo_cat.filter(col('TOTAL_NBRX_C3') > 0)
print("Shape :", non_zero_train_data.count(), len(non_zero_train_data.columns))

non_zero_train_data = non_zero_train_data.toPandas()
X_nz = non_zero_train_data.drop('TOTAL_NBRX_C3')
y_nz = non_zero_train_data['TOTAL_NBRX_C3']

mi_scores = mutual_info_regression(X_nz, y_nz)
mi_series = pd.Series(mi_scores, index=non_zero_train_data.columns)

# Select top N features
top_n = 3000
top_features = mi_series.sort_values(ascending=False).head(top_n).index
top_features.display()

In [0]:
train_data_wo_cat = train_data_wo_cat.select(*top_features + ['TOTAL_NBRX_C3'])
print("Shape :", train_data_wo_cat.count(), len(train_data_wo_cat.columns))

#### Step 3: XGB/LGBM Feature Selection

In [0]:
# Sampling the data to 10% of overall size for selection
train_data_wo_cat_sampled = train_data_wo_cat.sample(withReplacement=False, fraction=0.1, seed=42)
train_data_wo_cat_sampled = train_data_wo_cat_sampled.toPandas()

print("Shape :", train_data_wo_cat_sampled.shape)

In [0]:
from xgboost import XGBRegressor

def xgb_feature_selection(X, y, top_k=1000):
    model = XGBRegressor(n_estimators=300, max_depth=5, n_jobs=-1, random_state=42)
    model.fit(X, y)
    importances = model.feature_importances_
    
    # Select top-k features
    feature_series = pd.Series(importances, index=X.columns)
    selected_cols = feature_series.sort_values(ascending=False).head(top_k).index.tolist()
    return selected_cols

In [0]:
xgb_selected_cols = xgb_feature_selection(train_data_wo_cat_sampled.drop('TOTAL_NBRX_C3'), train_data_wo_cat_sampled['TOTAL_NBRX_C3'])
pd.DataFrame(xgb_selected_cols).display()

In [0]:
pd.DataFrame(xgb_selected_cols).to_csv(
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/xgb_selected_cols.csv",
    index=False
)

In [0]:
from lightgbm import LGBMRegressor

def lgbm_feature_selection(X, y, top_k=1000):
    model = LGBMRegressor(n_estimators=300, max_depth=5, n_jobs=-1, random_state=42)
    model.fit(X, y)
    importances = model.feature_importances_
    
    # Select top-k features
    feature_series = pd.Series(importances, index=X.columns)
    selected_cols = feature_series.sort_values(ascending=False).head(top_k).index.tolist()
    return selected_cols

In [0]:
lgbm_selected_cols = lgbm_feature_selection(train_data_wo_cat_sampled.drop('TARGET'), train_data_wo_cat_sampled['TARGET'])
pd.DataFrame(lgbm_selected_cols).display()

In [0]:
# lgbm_selected_cols save list of features




pd.DataFrame(xgb_selected_cols).to_csv(
    "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/lgbm_selected_cols.csv",
    index=False
)